# PCA-Based Face Similarity Search (Eigenfaces)

This notebook builds a face **similarity search** system using classic
**eigenfaces**: PCA fit on grayscale face images, then nearest-neighbor
search in that compressed PCA space using three different distance
metrics -- **Euclidean**, **Cosine**, and **Manhattan** -- compared side by
side.

**Dataset:** the Olivetti Faces dataset (AT&T Laboratories Cambridge) --
400 grayscale 64x64 images of 40 people, 10 images each, varying lighting,
expression, and glasses/no-glasses. Loaded via
`sklearn.datasets.fetch_olivetti_faces()`, which downloads a small archive
on first run and caches it locally afterwards (no manual download needed).

### Pipeline
1. Load the dataset & look at some faces
2. Fit PCA ("eigenfaces") on the flattened pixel vectors
3. Visualize the top eigenfaces and cumulative explained variance
4. Reconstruct faces from a reduced number of components (compression demo)
5. Similarity search: given a query face, rank every other face by
   Euclidean / Cosine / Manhattan distance in eigenface space
6. Evaluate retrieval quality per metric with a leave-one-out top-k accuracy
   (does the nearest match actually belong to the same person?)


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sys, os
sys.path.append(os.path.abspath("../src"))

import numpy as np
import matplotlib.pyplot as plt

from face_similarity import (
    load_faces, fit_eigenfaces, project,
    all_metrics, rank_by_metric, leave_one_out_accuracy, IMAGE_SHAPE
)

plt.rcParams["figure.facecolor"] = "#0b0d12"
plt.rcParams["axes.facecolor"] = "#12151c"
plt.rcParams["savefig.facecolor"] = "#0b0d12"
plt.rcParams["text.color"] = "#e6e6e6"
plt.rcParams["axes.labelcolor"] = "#e6e6e6"
plt.rcParams["xtick.color"] = "#cccccc"
plt.rcParams["ytick.color"] = "#cccccc"
plt.rcParams["axes.edgecolor"] = "#444444"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 1. Load the dataset

`images` is `(400, 64, 64)` for display, `data` is the same images flattened
to `(400, 4096)` for modeling, and `target` holds the person ID (0-39) for
each image -- used later ONLY to evaluate retrieval quality, never as a
model input (this is a similarity-search system, not a classifier).


In [ ]:
images, data, target = load_faces(shuffle=False)
print(f"images: {images.shape}   data: {data.shape}   target: {target.shape}")
print(f"People: {len(np.unique(target))}   Images per person: {data.shape[0] // len(np.unique(target))}")


In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i * 5], cmap="gray")
    ax.set_title(f"Person {target[i*5]}", fontsize=9)
    ax.axis("off")
fig.suptitle("Sample faces (one per several people)", color="#e6e6e6")
plt.tight_layout()
plt.show()


## 2. Fit PCA ("eigenfaces")

Each 64x64 image is a point in 4,096-dimensional pixel space. PCA finds the
directions of greatest variation across all 400 faces -- the **eigenfaces**
-- and lets us represent every face with far fewer numbers while keeping
95% of the meaningful variation between faces.


In [ ]:
pca = fit_eigenfaces(data, n_components=0.95, whiten=True)
embeddings = project(pca, data)

print(f"Original dimensionality: {data.shape[1]}")
print(f"Eigenfaces kept (95% variance): {pca.n_components_}")
print(f"Compression ratio: {data.shape[1] / pca.n_components_:.1f}x")


In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(pca.components_[i].reshape(IMAGE_SHAPE), cmap="gray")
    ax.set_title(f"Eigenface {i+1}", fontsize=9)
    ax.axis("off")
fig.suptitle("Top 16 eigenfaces", color="#e6e6e6")
plt.tight_layout()
plt.show()


In [ ]:
cum_var = np.cumsum(pca.explained_variance_ratio_)
plt.figure(figsize=(8, 4))
plt.plot(cum_var, color="#f5c542")
plt.axhline(0.95, color="#ff4d4d", linestyle="--", linewidth=1, label="95% variance")
plt.axvline(pca.n_components_, color="#6c63ff", linestyle="--", linewidth=1)
plt.xlabel("Number of eigenfaces")
plt.ylabel("Cumulative explained variance")
plt.title("How many eigenfaces do we actually need?")
plt.legend()
plt.tight_layout()
plt.show()


## 3. Reconstruction demo

Projecting a face into eigenface space and back shows how much detail
survives the compression -- this is exactly the representation the
similarity search below operates on.


In [ ]:
sample_idx = 12
reconstructed = pca.inverse_transform(embeddings[sample_idx].reshape(1, -1)).reshape(IMAGE_SHAPE)

fig, axes = plt.subplots(1, 2, figsize=(6, 3.2))
axes[0].imshow(images[sample_idx], cmap="gray")
axes[0].set_title("Original", fontsize=10)
axes[0].axis("off")
axes[1].imshow(reconstructed, cmap="gray")
axes[1].set_title(f"Reconstructed ({pca.n_components_}-D)", fontsize=10)
axes[1].axis("off")
plt.tight_layout()
plt.show()


## 4. Similarity search with three distance metrics

Given a query face, every other face in the dataset is scored by:

| Metric | Interpretation |
|---|---|
| **Euclidean** | straight-line distance in eigenface space — general purpose, sensitive to overall magnitude differences |
| **Cosine** | angle between eigenface vectors — captures *shape* of the face's representation regardless of overall brightness/contrast |
| **Manhattan** | sum of absolute differences — more robust to a few large outlier dimensions than Euclidean |

Lower Euclidean/Manhattan = more similar. Higher Cosine = more similar.


In [ ]:
query_idx = 7   # pick any face 0-399 to try
gallery_idx = np.delete(np.arange(len(embeddings)), query_idx)
query_vec = embeddings[query_idx]
gallery_vecs = embeddings[gallery_idx]

top_n = 5
results_by_metric = {}
for metric in ["euclidean", "cosine", "manhattan"]:
    ranked = rank_by_metric(query_vec, gallery_vecs, metric, top_n=top_n)
    for r in ranked:
        r["true_index"] = gallery_idx[r["index"]]
        r["person"] = target[r["true_index"]]
    results_by_metric[metric] = ranked

print(f"Query: face #{query_idx}, person {target[query_idx]}\n")
for metric, ranked in results_by_metric.items():
    print(f"Top-{top_n} matches by {metric}:")
    for rank, r in enumerate(ranked, 1):
        match = "✓" if r['person'] == target[query_idx] else " "
        print(f"  {rank}. face #{r['true_index']:>3}  person {r['person']:>2}  {match}  "
              f"(euc={r['euclidean']:.3f} cos={r['cosine']:.4f} man={r['manhattan']:.3f})")
    print()


In [ ]:
fig, axes = plt.subplots(3, top_n + 1, figsize=(14, 7))
for row, metric in enumerate(["euclidean", "cosine", "manhattan"]):
    axes[row, 0].imshow(images[query_idx], cmap="gray")
    axes[row, 0].set_title("QUERY", fontsize=9, color="#f5c542")
    axes[row, 0].axis("off")
    for col, r in enumerate(results_by_metric[metric], start=1):
        axes[row, col].imshow(images[r["true_index"]], cmap="gray")
        border_color = "#4bb28a" if r["person"] == target[query_idx] else "#ff4d4d"
        axes[row, col].imshow(images[r["true_index"]], cmap="gray")
        axes[row, col].set_title(f"#{r['true_index']} (p{r['person']})", fontsize=8, color=border_color)
        axes[row, col].axis("off")
    axes[row, 0].set_ylabel(metric, fontsize=11, color="#e6e6e6")

for row, metric in enumerate(["euclidean", "cosine", "manhattan"]):
    axes[row, 0].text(-25, 32, metric.capitalize(), rotation=90, va="center", fontsize=11, color="#e6e6e6")

plt.suptitle("Top matches per metric (green title = correct person, red = different person)", color="#e6e6e6")
plt.tight_layout()
plt.show()


## 5. Which metric actually retrieves the right person?

Since we know each face's true identity, we can evaluate retrieval quality
directly: for every face, treat it as a query against every *other* face and
check whether the top-k matches include the same person. This never touches
model training -- it is a pure evaluation step, exactly like checking
labels against an unsupervised clustering result.


In [ ]:
metrics = ["euclidean", "cosine", "manhattan"]
ks = [1, 3, 5]

results_table = {m: [] for m in metrics}
for metric in metrics:
    for k in ks:
        acc = leave_one_out_accuracy(embeddings, target, metric, k=k)
        results_table[metric].append(acc)

import pandas as pd
df = pd.DataFrame(results_table, index=[f"top-{k}" for k in ks]).T
df.columns = [f"Top-{k} accuracy" for k in ks]
df


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
colors = {"euclidean": "#3b6ea5", "cosine": "#f5c542", "manhattan": "#4bb28a"}
for metric in metrics:
    ax.plot(ks, results_table[metric], marker="o", label=metric, color=colors[metric])
ax.set_xticks(ks)
ax.set_xlabel("k (top-k retrieved)")
ax.set_ylabel("Accuracy (same person retrieved)")
ax.set_title("Retrieval accuracy by metric")
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.show()


## 6. Where to go from here

- **`src/face_similarity.py`** — all the reusable logic above (loading,
  eigenfaces, the three metrics, retrieval evaluation) is shared with the
  Streamlit app below, so both use exactly the same pipeline.
- **`app/app.py`** — a dark-themed Streamlit app: pick any face from the
  built-in gallery (or upload your own grayscale face photo), see the top-N
  matches ranked by whichever metric you choose, with all three metrics
  shown side by side for every result.
  ```
  streamlit run app/app.py
  ```
- To extend further: try a deep face-embedding model (e.g. FaceNet/ArcFace)
  as a fourth comparison point — eigenfaces are a great, fast, fully
  classical baseline, but deep embeddings typically retrieve the correct
  person far more reliably across lighting/pose changes.
